# Exercise 3: prompt experiment – Google Flow and OpenAI API

## Tasks

1. **One prompt:** write a prompt and create a picture in Flow.
2. **Vary:** change one building block of the prompt and create again.
3. **Repeat:** create several pictures for each version.
4. **Compare:** rate one feature of the pictures – by eye or with the OpenAI API.

Images in Flow are free. Do not create videos yet.

**This is the solution.** Try `prompt_experiment.ipynb` first.

## Setup

Select the **Python 3.11** kernel. The pictures are created in **Google Flow** (a web app, no code). The OpenAI API key is only needed for the last section.

In [ ]:
# Standard Python libraries.
import sys

# Libraries installed from requirements.txt.
import pandas as pd
from pydantic import BaseModel

# helper.py is in the main folder, one level up.
sys.path.append("..")

In [ ]:
# Functions from helper.py.
from helper import ask, find_images, show_images

## 1. Write a prompt from building blocks

Describe what should be visible. Copy the printed prompt into Flow and create a picture.

In [ ]:
# The building blocks of the prompt.
subject = "a red wooden rowboat"
setting = "on a mountain lake"
light = "at dawn, soft light"
composition = "wide shot"

# Prompt A: all blocks in one line.
prompt_a = f"{subject}, {setting}, {light}, {composition}"
print("A:", prompt_a)

## 2. Change one building block and create again

Only **one** block changes, the rest stays the same – so you know what caused the difference.

In [ ]:
# Change only the composition.
composition = "wide shot, boat in the left third"

# Prompt B: the same blocks, with the new composition.
prompt_b = f"{subject}, {setting}, {light}, {composition}"
print("B:", prompt_b)

## 3. Create three pictures per version and upload them

Use the same Flow image model and aspect ratio for all pictures. Upload them into the folder `flow_images/` next to this notebook and name them:

`variant_a_1`, `variant_a_2`, `variant_a_3`, `variant_b_1`, `variant_b_2`, `variant_b_3` (`.jpg` or `.png`)

In [ ]:
# The six expected pictures.
names = [
    "variant_a_1", "variant_a_2", "variant_a_3",
    "variant_b_1", "variant_b_2", "variant_b_3",
]

# Find the files. This stops if a picture is missing.
paths = find_images("flow_images", names)

## 4. Compare the versions by eye

The feature to rate: **is the boat in the left third of the picture?**

In [ ]:
# Version A in the top row, version B in the bottom row.
show_images(paths, columns=3)

In [ ]:
# One row per picture.
ratings = pd.DataFrame()
ratings["name"] = names
ratings["version"] = ["A", "A", "A", "B", "B", "B"]

# Your rating per picture: True if the boat is in the left third.
# Replace these example values with your own.
ratings["me"] = [False, True, False, True, True, False]
ratings

In [ ]:
# Share of pictures with the feature, per version.
# True counts as 1 and False as 0, so the mean is the share of True.
ratings.groupby("version")["me"].mean()

## 5. Let the OpenAI API rate the same feature

The same yes/no question for all six pictures, sent to the **OpenAI API** (model `gpt-6-luna`). Does it agree with you?

In [ ]:
# The answer format: yes or no, plus a short reason.
class Rating(BaseModel):
    matches: bool
    reason: str

In [ ]:
# The same question for every picture.
PROMPT = """
Is the center of the red rowboat in the left third of the image?
If there is no boat, answer false.
Give a short reason.
"""

In [ ]:
# Ask the model about every picture.
api_ratings = []
for path in paths:
    answer = ask(PROMPT, Rating, [path])
    api_ratings.append(answer["matches"])

# Your rating and the OpenAI rating side by side.
ratings["api"] = api_ratings
ratings

In [ ]:
# Share of pictures with the feature, per version.
ratings.groupby("version")[["me", "api"]].mean()

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. Why does the same prompt give different pictures – and what does that mean for your experiment?
2. Why do we change only one building block between version A and version B?
3. With three pictures per version, the share can only be 0, 1/3, 2/3 or 1. What does this mean for your conclusion?
4. Where do your rating and the OpenAI rating differ? Who is right – and why can you not simply trust the API rating?
5. Flow has no seed setting. What do you write down so that someone else can follow your experiment?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. Generation starts from random noise, so every run is different. One picture per version proves nothing. You need several pictures per version and compare the shares.

2. If several blocks change, you cannot tell which one caused the difference. One change per round makes it an experiment instead of trial and error.

3. Three pictures are weak evidence: a single picture moves the share by a third. Describe the result as a tendency. A clear conclusion needs more pictures.

4. Look at the pictures where the two ratings differ. Models are weak at left/right and positions. Your own judgement is the reference, so always check a sample by hand.

5. The prompt, the model, the settings (e.g. aspect ratio) and the date. Keep all pictures – also the ones that did not work.

</details>

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')